# BAC-CAB identity — definition and application

The **BAC-CAB rule** is the vector triple product identity:

$$\mathbf{a} \times (\mathbf{b} \times \mathbf{c})
  = \mathbf{b}(\mathbf{a}\cdot\mathbf{c}) - \mathbf{c}(\mathbf{a}\cdot\mathbf{b})$$

This notebook demonstrates:
1. Defining a named algebraic identity with pattern variables.
2. Applying it manually to a concrete expression.
3. Using automatic pattern matching (`find_matches` / `apply_identity_auto`).
4. The standard library entry for the same identity.

> **Prerequisite:** build the Python bindings and set `PYTHONPATH`, or run
> `source examples/env.sh` before launching Jupyter.

In [ ]:
import pathlib
from IPython.display import display, Math

from tender import (
    tensor,
    make_pattern_var,
    cross,
    dot,
    Identity,
    apply_identity,
    find_matches,
    apply_identity_auto,
    doc,
    State,
    Derivation,
    show_jupyter,
    to_latex_document,
    Contract,
    Sum,
)

## Define the BAC-CAB identity

Pattern variables act as typed placeholders. `constrain_rank(1)` means the
variable stands for any rank-1 (vector) expression.

In [ ]:
a = make_pattern_var("a"); a.constrain_rank(1)
b = make_pattern_var("b"); b.constrain_rank(1)
c = make_pattern_var("c"); c.constrain_rank(1)

lhs = cross(a, cross(b, c))
rhs = b * dot(a, c) - c * dot(a, b)

bac_cab = Identity("BAC-CAB", lhs, rhs)
doc(bac_cab, format="jupyter")

## Apply to a concrete expression

Target: $\mathbf{u} \times (\mathbf{v} \times \mathbf{w})$

In [ ]:
u = tensor(r"\mathbf{u}", 1)
v = tensor(r"\mathbf{v}", 1)
w = tensor(r"\mathbf{w}", 1)

expr = cross(u, cross(v, w))

step = apply_identity(bac_cab, {a: u, b: v, c: w})
history = Derivation([step]).apply(State(expr))

show_jupyter(history)

## Automatic pattern matching

`find_matches` scans the expression tree for all sites where the identity LHS
matches. `apply_identity_auto` picks the first match and applies the identity.

In [ ]:
matches = find_matches(bac_cab, expr)
print(f"{len(matches)} match(es) found")

auto_step = apply_identity_auto(bac_cab, expr)
auto_history = Derivation([auto_step]).apply(State(expr))

show_jupyter(auto_history)

## Standard library

`tender.lib` ships the BAC-CAB rule (and other identities) ready to import.

In [ ]:
from tender.lib.identities.epsilon import bac_cab as lib_bac_cab

doc(lib_bac_cab, format="jupyter")

## Verification

In [ ]:
result = history[-1].expr

assert isinstance(result, Sum), f"Expected Sum, got {type(result).__name__}"
assert len(result.terms) == 2, f"Expected 2 terms, got {len(result.terms)}"

def has_contract(e):
    if isinstance(e, Contract): return True
    if hasattr(e, "expr"):      return has_contract(e.expr)    # Scale
    if hasattr(e, "lhs"):       return has_contract(e.lhs) or has_contract(e.rhs)
    return False

assert all(has_contract(t) for t in result.terms)
assert len(matches) == 1
assert auto_history[-1].expr.latex() == history[-1].expr.latex()

print("All assertions passed.")

## Export to PDF

Write a compilable LaTeX document to `out/bac_cab.tex`.
Or simply run `make bac_cab` from the `examples/` directory.

In [ ]:
tex = to_latex_document(history, title="BAC-CAB identity — application")
out_dir = pathlib.Path("out")
out_dir.mkdir(exist_ok=True)
(out_dir / "bac_cab.tex").write_text(tex)
print("Written to out/bac_cab.tex")
print("Compile with: pdflatex -output-directory out out/bac_cab.tex")